In [1]:
from fastai.vision.all import *
from pathlib import Path
import pandas as pd


def find_base_path():
    candidates = [
        Path("/kaggle/input/aerial-cactus-identification"),
        Path("../input/aerial-cactus-identification"),
        Path("input/aerial-cactus-identification"),
        Path("."),
    ]
    for p in candidates:
        if (p / "train.csv").exists() and (p / "sample_submission.csv").exists():
            return p
    raise FileNotFoundError("Base data directory not found.")


base_path = find_base_path()
train_dir = base_path / "train"
test_dir = base_path / "test"

train_df = pd.read_csv(base_path / "train.csv")
test_df = pd.read_csv(base_path / "sample_submission.csv")



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
dls = ImageDataLoaders.from_df(
    df=train_df,
    path=base_path,
    folder="train",
    fn_col="id",
    label_col="has_cactus",
    valid_pct=0.01,
    seed=42,
    item_tfms=Resize(128),
    batch_tfms=aug_transforms(
        do_flip=True,
        flip_vert=True,
        max_rotate=10.0,
        max_zoom=1.1,
        max_lighting=0.2,
        max_warp=0.2,
        p_affine=0.75,
        p_lighting=0.75,
    ),
    bs=64,
    device=default_device(),
)

learn1 = cnn_learner(dls, models.densenet161, metrics=[error_rate, accuracy])
lr = 1e-2
learn1.fit_one_cycle(3, lr)

# Build test dataloader using absolute paths to avoid FileNotFoundError
test_items = [test_dir / fname for fname in test_df["id"]]
test_dl = learn1.dls.test_dl(test_items, with_labels=False)
preds1, _ = learn1.get_preds(dl=test_dl, reorder=False)



/usr/local/lib/python3.11/dist-packages/fastai/vision/learner.py:303: UserWarning: `cnn_learner` has been renamed to `vision_learner` -- please update your code
  warn("`cnn_learner` has been renamed to `vision_learner` -- please update your code")
Downloading: "https://download.pytorch.org/models/densenet161-8d451a50.pth" to /root/.cache/torch/hub/checkpoints/densenet161-8d451a50.pth
100%|██████████| 110M/110M [00:01<00:00, 111MB/s]


epoch,train_loss,valid_loss,error_rate,accuracy,time
0,0.096385,0.025940,0.007092,0.992908,00:25
1,0.016829,0.000154,0.000000,1.000000,00:25
2,0.005516,0.000160,0.000000,1.000000,00:25


In [3]:
learn2 = cnn_learner(dls, models.resnet50, metrics=[error_rate, accuracy])
lr2 = 1e-2
learn2.fit_one_cycle(3, lr2)

# Build test dataloader for the second model, also using absolute paths
test_items2 = [test_dir / fname for fname in test_df["id"]]
test_dl2 = learn2.dls.test_dl(test_items2, with_labels=False)
preds2, _ = learn2.get_preds(dl=test_dl2, reorder=False)



/usr/local/lib/python3.11/dist-packages/fastai/vision/learner.py:303: UserWarning: `cnn_learner` has been renamed to `vision_learner` -- please update your code
  warn("`cnn_learner` has been renamed to `vision_learner` -- please update your code")
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 113MB/s]


epoch,train_loss,valid_loss,error_rate,accuracy,time
0,0.120164,0.020265,0.014184,0.985816,00:11
1,0.053119,0.000852,0.000000,1.000000,00:11
2,0.032539,0.003363,0.000000,1.000000,00:11


In [4]:
# Average the cactus probabilities from both models
avg_cactus_prob = 0.5 * (preds1[:, 1] + preds2[:, 1])
submission = test_df.copy()
submission["has_cactus"] = avg_cactus_prob.cpu().numpy()
# Write the submission file with the required .csv suffix
submission.to_csv("/kaggle/working/submission.csv", index=False)